In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import DistilBertTokenizerFast
from transformers import DistilBertModel
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import Adam
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt



class DisasterTweetDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}, torch.tensor(self.labels[idx])

    def __len__(self):
        return len(self.labels)



class DisasterClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        self.classifier = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]  # CLS token
        return torch.sigmoid(self.classifier(pooled_output)).squeeze()


class TestDataset(torch.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __getitem__(self, idx):
        return {key: val[idx] for key, val in self.encodings.items()}

    def __len__(self):
        return len(self.encodings["input_ids"])

In [ ]:
df = pd.read_csv("/kaggle/input/nlp-getting-started/train.csv")
texts = df["text"].tolist()
labels = df["target"].tolist()


train_texts, val_texts, train_labels, val_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

train_encodings = tokenizer(train_texts, truncation=True, padding=True, return_tensors="pt")
val_encodings = tokenizer(val_texts, truncation=True, padding=True, return_tensors="pt")

train_dataset = DisasterTweetDataset(train_encodings, train_labels)
val_dataset = DisasterTweetDataset(val_encodings, val_labels)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = DisasterClassifier().to(device)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

optimizer = Adam(model.parameters(), lr=1e-5)
loss_fn = nn.BCELoss()

for epoch in range(3):
    model.train()
    for batch in train_loader:
        inputs, labels = batch
        input_ids = inputs["input_ids"].to(device)
        attention_mask = inputs["attention_mask"].to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

    # Validation
    model.eval()
    preds, actuals = [], []
    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch
            input_ids = inputs["input_ids"].to(device)
            attention_mask = inputs["attention_mask"].to(device)
            outputs = model(input_ids, attention_mask)
            preds.extend(outputs.cpu().numpy())
            actuals.extend(labels.numpy())

    binary_preds = [1 if p > 0.5 else 0 for p in preds]
    acc = accuracy_score(actuals, binary_preds)
    print(f"Epoch {epoch+1}: Val Accuracy = {acc:.4f}")

    # Create confusion matrix
    cm = confusion_matrix(actuals, binary_preds)
    tn, fp, fn, tp = confusion_matrix(actuals, binary_preds).ravel()
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1_score = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Display it
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Disaster", "Disaster"])
    disp.plot(cmap=plt.cm.Blues)
    plt.title(f"Confusion Matrix - Epoch {epoch+1}  || f1 score {f1_score:.2f}")
    plt.show()

test_df = pd.read_csv("/kaggle/input/nlp-getting-started/test.csv")
texts = test_df["text"].tolist()

test_encodings = tokenizer(texts, truncation=True, padding=True, return_tensors="pt")
test_dataset = TestDataset(test_encodings)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=16)

model.eval()
predictions = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        outputs = model(input_ids, attention_mask)

        predictions.extend(outputs.cpu().numpy())

binary_preds = [1 if p > 0.5 else 0 for p in predictions]
test_df["target"] = binary_preds
test_df[["id", "target"]].to_csv("/kaggle/working/submission.csv", index=False)